In [ ]:
# OLD PRETOKENIZER that needs to download the data from hf first
# use for instruction response sets
# DO NOT use if your tokenizer has no post processor with added special tokens 
# It needs to be capping each seq with an end of text token

import os
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
from tokenizers import Tokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

tokenizer_file = '/home/user/Desktop/Python_Scripts/gpt_course/gpt_dt/tokenizer/tokenizer-01.json'
HF_DATASET = "microsoft/orca-math-word-problems-200k"
block_size = 512
tokenizer = Tokenizer.from_file(tokenizer_file)

In [14]:
dataset = load_dataset(HF_DATASET, num_proc=12) # , split="train"
dataset

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/84.2M [00:00<?, ?B/s]

Setting num_proc from 12 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Generating train split:   0%|          | 0/200035 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 200035
    })
})

In [15]:
dataset = dataset["train"]
seed_value = 42

dataset = dataset.train_test_split(
    test_size=0.10,
    seed=seed_value
)
dataset.flatten()
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 180031
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 20004
    })
})

In [ ]:
#edit tokenize function for your data feilds
def tokenize_function(examples):
    batch_pairs = [
        (inp, out) 
        for inp, out in zip(examples["question"], examples["answer"])
    ]
    outputs = tokenizer.encode_batch(batch_pairs) # adds your special tokens and eot id at the end

    final_ids = []
    for o in outputs:
        final_ids.append(o.ids)

    return {"ids": final_ids}


for split, dset in dataset.items():
    
    tokenized_dset = dset.map(
        tokenize_function,
        batched=True,
        num_proc=12,
        batch_size=1000,
        keep_in_memory=False, 
        load_from_cache_file=False,
        remove_columns=dset.column_names, 
        desc=f"Tokenizing {split} split"
    )

    filename = f"{HF_DATASET}_{split}_data.bin"
    
    print(f"Calculating total length for {filename}...")
    total_len = len(tokenized_dset)

    # Using uint16 for efficiency (works for vocab < 65,535)
    arr = np.memmap(filename, dtype=np.uint16, mode='w+', shape=(total_len,))
    
    idx = 0
    for example in tqdm(tokenized_dset["ids"], desc=f"Writing {filename}"):
        arr[idx : idx + len(example)] = example
        idx += len(example)
        
    arr.flush()
    print(f"Finished {filename}. Total tokens: {total_len}")

Tokenizing train split (num_proc=12):   0%|          | 0/180031 [00:00<?, ? examples/s]

Calculating total length for lamini_train_data.bin...


Writing lamini_train_data.bin: 100%|█████████████████████████████████████████████| 152807/152807 [00:12<00:00, 12686.33it/s]

Finished lamini_train_data.bin. Total tokens: 78237184
Lost samples: 27224


Tokenizing test split (num_proc=12):   0%|          | 0/20004 [00:00<?, ? examples/s]

Calculating total length for lamini_test_data.bin...


Writing lamini_test_data.bin: 100%|████████████████████████████████████████████████| 16937/16937 [00:01<00:00, 12686.02it/s]

Finished lamini_test_data.bin. Total tokens: 8671744
Lost samples: 3067


Done!
